In [17]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [21]:
from src import fetch
import pandas as pd
import json

data=fetch.get_json('https://www.rebas.tw/api/formal/players/PtGe1/seasons/CPBL-2026-oB/logs')['data']
print(f'{len(data)}場')
game=data[0]
print(game.keys())
pa=game['PA_list'][0]
print(pa)
print({k:v for k, v in pa.items() if k not in ('events','ball_counts')})

25場
dict_keys(['PA_list', 'away', 'date', 'home', 'location_abbr', 'opponent_abbr', 'opponent_uniqid', 'pitching', 'seq', 'strikezone_completeness', 'uniqid'])
{'LI': '', 'PA_order': 6, 'PA_round': 3, 'RBI': 0, 'RE': '0.504', 'RE24': '-0.239', 'SP_game_score': '', 'WPA': '-0.035', 'away_score': 0, 'b_hand': 'L', 'ball_counts': ['S', 'H'], 'balls': 0, 'bases': 0, 'batter': {'name': '高捷', 'number': '4', 'uniqid': 'SCI7M'}, 'endOuts': 1, 'end_away_score': 0, 'end_home_score': 0, 'events': [{'in_play': False, 'is_ball': False, 'is_scoring': False, 'is_strike': False, 'pitch': None, 'type': 'SUB'}, {'in_play': False, 'is_ball': False, 'is_scoring': False, 'is_strike': True, 'pitch': {'EV': '', 'RPM': '2449', 'code': 'S', 'coord_x': '57.78', 'coord_y': '22.08', 'fielder_position': '', 'hardness': '', 'location_code': '', 'location_coord_x': '', 'location_coord_y': '', 'trajectory': '', 'type': 'FC', 'velocity': '132'}, 'type': 'PITCH'}, {'in_play': True, 'is_ball': False, 'is_scoring': False

In [20]:
%%writefile -a src/fetch.py

# ---------- Day 8：逐打席紀錄 ----------
pa_path=Path(__file__).resolve().parent.parent/"data"/"raw"/"pa"

def flatten_pa(logs):
    rows=[]
    for game in logs:
        for pa in game.get('PA_list',[]):
            pitches=[e['pitch'] for e in pa.get('events',[]) if e.get('type')=='PITCH' and e.get('pitch')]
            rows.append({
                'game_id':game['uniqid'],
                'date':game['date'],
                'inning':pa.get('inning'),
                'pa_round':pa.get('PA_round'),
                'batter_id':pa['batter']['uniqid'],
                'batter':pa['batter']['name'],
                'pitcher_id':pa['pitcher']['uniqid'],
                'pitcher':pa['pitcher']['name'],
                'b_hand':pa.get('b_hand'),
                'p_hand':pa.get('p_hand'),
                'result':pa.get('result'),
                'pitch_types':','.join(p.get('type') or '' for p in pitches),
                'pitch_codes':','.join(p.get('code') or '' for p in pitches)
            })
    return rows


Appending to src/fetch.py


In [22]:
%%writefile -a src/fetch.py

def fetch_pitcher_pa(player_id,season=SEASON,refresh=False):
    pa_path.mkdir(parents=True,exist_ok=True)
    file=pa_path/f'{season}_{player_id}.csv'
    if file.exists() and not refresh:
        df=pd.read_csv(file,dtype=str,keep_default_na=False)
    else:
        data_list=get_json(f'https://www.rebas.tw/api/formal/players/{player_id}/seasons/{season}/logs')['data']
        df=pd.DataFrame(flatten_pa(data_list)).astype(str)
        df.to_csv(file,index=False)
    # 野手兼投手（例如林子偉）的紀錄裡也有他當打者的打席，只留他當投手的
    return df[df['pitcher_id']==player_id]


Appending to src/fetch.py


In [23]:
%%writefile scripts/fetch_all_pa.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import pandas as pd
from src import fetch

pitchers=fetch.fetch_player('pitcher',refresh=False)
frames=[]
for i , pid in enumerate(pitchers['player_id'],1):
    frames.append(fetch.fetch_pitcher_pa(pid))
    if i % 20 == 0:
        print(f'{i}/{len(pitchers)}')

df_all_pitcher=pd.concat(frames,ignore_index=True)
out=Path(__file__).resolve().parent.parent/ "data" / "clean" / "pa.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df_all_pitcher.to_csv(out,index=False)
print(f'完成：{len(df_all_pitcher)} 個打席 → {out}')


Writing scripts/fetch_all_pa.py


In [5]:
import pandas as pd
csv=pd.read_csv('data/clean/pa.csv',dtype=str,keep_default_na=False)

print(len(csv))
print(csv['result'].value_counts().head(5))
print(csv["b_hand"].value_counts())

27268
result
FO     7165
GO     4889
1B     4768
SO     4319
uBB    2108
Name: count, dtype: int64
b_hand
L    15535
R    11733
Name: count, dtype: int64
